<a href="https://colab.research.google.com/github/josevargasmercado02/Proyectos_Fisica_Computacional/blob/main/Caso4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Cuerda con extremo libre forzado
### Modos normales, resonancia y simulación numérica

Este notebook estudia las **vibraciones transversales de una cuerda** de longitud $L$ con un extremo fijo ($x=0$) y el otro extremo libre sometido a una fuerza armónica externa ($x=L$). El problema se resuelve de dos maneras complementarias:

1. **Problema de valores propios** (método de Jacobi implementado desde cero) para obtener las frecuencias naturales y los modos normales.
2. **Simulación en el dominio del tiempo** (diferencias finitas explícitas) para obtener la curva de resonancia y validarla contra los resultados anteriores y contra la solución analítica.


## 1. Modelo físico

La cuerda, de densidad lineal de masa $\mu$ y tensión $T$, obedece la **ecuación de onda**:

$$
\mu \frac{\partial^2 u}{\partial t^2} = T \frac{\partial^2 u}{\partial x^2}
$$

**Condiciones de frontera**

| Extremo | Condición |
|---|---|
| $x = 0$ (fijo) | $u(0,t) = 0$ |
| $x = L$ (libre, forzado) | $T \dfrac{\partial u}{\partial x}\Big|_{x=L} = F_0 \cos(\omega t)$ |

**Solución analítica (frecuencias naturales).** Para fijo–libre, las frecuencias angulares de resonancia son:

$$
\omega_n = \frac{(2n-1)\,\pi}{2L}\sqrt{\frac{T}{\mu}}, \qquad n = 1, 2, 3, \dots
$$

**Discretización espacial.** Con $N$ intervalos de tamaño $\Delta x = L/N$, el problema de modos normales se escribe como un problema generalizado de valores propios

$$
A\,\mathbf{x} = \omega^2 M\,\mathbf{x}, \qquad A = \frac{T}{\Delta x^2}\,\mathrm{tridiag}(-1,\,2,\,-1)
$$

donde el último nodo (extremo libre) tiene **la mitad de la masa** y $A_{NN}=T/\Delta x^2$ por simetría. Para obtener un problema simétrico estándar se define $D = M^{-1/2}$ y

$$
C = D\,A\,D, \qquad C\,\mathbf{y} = \omega^2\,\mathbf{y}, \qquad \mathbf{x} = D\,\mathbf{y}.
$$

**Integración temporal.** Se usa el esquema explícito de diferencias finitas centradas, estable si el número de Courant cumple

$$
\mathcal{C} = \frac{\Delta t}{\Delta x}\sqrt{\frac{T}{\mu}} \le 1 .
$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from scipy.signal import find_peaks

In [ ]:
# Parámetros
L = 1.0     # Largo de la cuerda
T = 1.0     # Tensión
mu = 2.0   # Densidad de masa
N = 10     # Número de nodos
F_0 = 0.001     # Amplitud de la fuerza
dt = 0.005      # Paso temporal
tiempo_total = 30.0 # Tiempo total

## 3. Frecuencias naturales: método de Jacobi

### 3.1 Construcción de las matrices

`construir_matrices` arma la matriz de masa $M$ (con la mitad de masa en el extremo libre), la matriz de rigidez $A$ y la matriz simétrica transformada $C = DAD$.

In [ ]:
def construir_matrices(L, T, mu, N):
    dx = L / N  # Separación entre nodos

    # Vector de masa efectiva
    mu_vector = mu * np.ones(N)
    # El nodo en el extremo libre (el último) tiene la mitad de la masa por el balance de simetría
    mu_vector[-1] = mu / 2.0

    # Matriz M
    M = np.diag(mu_vector)

    # Construcción de la matriz A
    d_p = 2.0 * np.ones(N) # Diagonal principal
    d_p[-1] = 1.0          # Modificación clave para el extremo libre (simetría)
    d_s = -1.0 * np.ones(N - 1) # Diagonal secundaria

    A = (T / (dx**2)) * (np.diag(d_p) + np.diag(d_s, k=1) + np.diag(d_s, k=-1))

    # Matriz D y matriz simétrica C
    D = np.diag(1.0 / np.sqrt(mu_vector))
    C = D @ A @ D

    return C, D

### 3.2 Método de Jacobi

El método de Jacobi diagonaliza una matriz simétrica mediante **rotaciones planas sucesivas**. En cada iteración:

1. Se localiza el elemento fuera de la diagonal de mayor módulo, $C_{pq}$.
2. Se calcula el ángulo $\theta$ que anula ese elemento:
   $$\theta = \tfrac{1}{2}\arctan\!\left(\frac{2C_{pq}}{C_{pp}-C_{qq}}\right)$$
3. Se aplica la rotación ortogonal $C \leftarrow Q^{T} C\, Q$ y se acumula $V \leftarrow V Q$.

El proceso termina cuando $\max_{p\neq q}|C_{pq}| < \varepsilon$. La diagonal final contiene los valores propios $\omega_n^2$ y las columnas de $V$ los vectores propios.

In [ ]:
def metodo_jacobi(C_in, epsilon=1e-8, N_max=20000):
    N = C_in.shape[0] # Tamaño de la matriz
    C = C_in.copy()   # Copia para no alterar la matriz original
    V = np.eye(N)     # Matriz acumuladora de vectores propios (identidad inicial)

    for k in range(N_max):
        # Optimización: Buscar exclusivamente el valor máximo absoluto por fuera de la diagonal
        # utilizando la parte triangular superior de la matriz simétrica
        C_upper = np.triu(np.abs(C), k=1)
        idx = np.argmax(C_upper)
        p, q = np.unravel_index(idx, C.shape)

        max_val = np.abs(C[p, q])

        # Criterio de parada: si el máximo es casi cero, la matriz ya es diagonal
        if max_val < epsilon:
            break

        # Calcular el ángulo de rotación theta
        if C[p, p] == C[q, q]:
            theta = (np.pi / 4.0) * np.sign(C[p, q]) # Evitar división por cero
        else:
            theta = 0.5 * np.arctan(2.0 * C[p, q] / (C[p, p] - C[q, q]))

        # Construir la matriz de rotación ortogonal Q
        Q = np.eye(N)
        Q[p, p] = np.cos(theta)
        Q[q, q] = np.cos(theta)
        Q[p, q] = -np.sin(theta)
        Q[q, p] = np.sin(theta)

        # Actualizar C (se va diagonalizando) y V (acumula los vectores propios)
        C = Q.T @ C @ Q
        V = V @ Q

    valores_propios = np.diag(C) # La diagonal final contiene los valores propios
    return valores_propios, V

### 3.3 Resolución del problema completo

`resolver_cuerda` encadena todos los pasos: construye las matrices, aplica Jacobi, ordena los modos de menor a mayor frecuencia, calcula $\omega_n = \sqrt{\lambda_n}$ y recupera los modos físicos $X = DY$, añadiendo el nodo fijo $x_0 = 0$.

In [ ]:
def resolver_cuerda(L, T, mu, N):
    # 1. Construir matrices
    C, D = construir_matrices(L, T, mu, N)

    # 2. Valores y vectores propios del problema transformado
    valores_propios, Y = metodo_jacobi(C)

    # 3. Ordenar valores y vectores propios de menor a mayor
    indices_ordenados = np.argsort(valores_propios)
    valores_propios = valores_propios[indices_ordenados]
    Y = Y[:, indices_ordenados]

    # 4. Cálculo de frecuencias naturales
    frecuencias_angulares = np.sqrt(valores_propios)

    # 5. Recuperar modos físicos X = D * Y
    X_activos = D @ Y

    # 6. Incorporar condición de frontera
    # X_0 = 0 (extremo fijo).
    # El último nodo (extremo libre) YA está en X_activos y su amplitud NO es cero.
    # Creamos una matriz de (N+1) filas y N columnas.
    X_completos = np.zeros((N + 1, N))
    X_completos[1:, :] = X_activos # Llenamos desde la fila 1 en adelante

    return valores_propios, frecuencias_angulares, X_completos

In [ ]:
_, omegas_naturales, modos = resolver_cuerda(L, T, mu, N)

print("Primeras 3 Frecuencias angulares naturales (omega_n):")
for i in range(3):
    print(f"Modo {i+1}: {omegas_naturales[i]:.4f} rad/s")

## 4. Simulación temporal de la cuerda forzada

Se integra la ecuación de onda con diferencias finitas explícitas:

$$
u_i^{n+1} = 2u_i^{n} - u_i^{n-1} + \mathcal{C}^2\left(u_{i+1}^{n} - 2u_i^{n} + u_{i-1}^{n}\right)
$$

- **Extremo fijo:** $u_0^{n+1} = 0$.
- **Extremo libre forzado:** se introduce un nodo fantasma que incorpora la condición $T\,u_x = F_0\cos(\omega t)$:
  $$u_N^{n+1} = 2u_N^{n} - u_N^{n-1} + \mathcal{C}^2\left(2u_{N-1}^{n} - 2u_N^{n} + \frac{2\Delta x\,F_0}{T}\cos(\omega t_n)\right)$$

La función devuelve la **amplitud máxima del extremo libre** en la segunda mitad de la simulación (para descartar el transitorio inicial) y lanza un error si se viola la condición de estabilidad $\mathcal{C} \le 1$.

In [ ]:
def simular_cuerda_forzada(L, T, mu, N, F_0, omega, dt, tiempo_total):
    dx = L / N
    C = (dt / dx) * np.sqrt(T / mu)

    # Verificación estricta de estabilidad
    if C >= 1.0:
        raise ValueError("El paso temporal dt es demasiado grande. C debe ser <= 1.")

    pasos_temporales = int(tiempo_total / dt)

    # Arreglos de estado: u_pasado (n-1), u_presente (n), u_futuro (n+1)
    # Tamaño N+1 para incluir desde x=0 hasta x=L
    u_pasado = np.zeros(N + 1)
    u_presente = np.zeros(N + 1)
    u_futuro = np.zeros(N + 1)

    amplitud_maxima = 0.0

    for n in range(pasos_temporales):
        t = n * dt

        # 1. Ecuación de onda para nodos interiores (1 hasta N-1)
        u_futuro[1:-1] = (2 * u_presente[1:-1] - u_pasado[1:-1] +
                          C**2 * (u_presente[2:] - 2 * u_presente[1:-1] + u_presente[:-2]))

        # 2. Condición de frontera: Extremo fijo en x=0
        u_futuro[0] = 0.0

        # 3. Condición de frontera: Extremo forzado en x=L (índice N)
        u_futuro[N] = (2 * u_presente[N] - u_pasado[N] +
                       C**2 * (2 * u_presente[N-1] - 2 * u_presente[N] +
                               (2 * dx * F_0 / T) * np.cos(omega * t)))

        # Registrar amplitud máxima una vez que los transitorios iniciales han decaído
        if n > pasos_temporales * 0.5:
            amplitud_maxima = max(amplitud_maxima, np.abs(u_futuro[N]))

        # Actualizar estados para el siguiente paso temporal
        u_pasado[:] = u_presente[:]
        u_presente[:] = u_futuro[:]

    return amplitud_maxima

## 5. Curva de resonancia

Se barre la frecuencia de excitación $\omega \in [0.1,\,6.0]$ rad/s con 400 puntos y, para cada una, se simula la cuerda y se registra la amplitud máxima. Los **picos de la curva** (detectados con `scipy.signal.find_peaks`) corresponden a las frecuencias de resonancia.

In [ ]:
# Definimos el intervalo de frecuencias (barrido)
# Usamos 400 puntos para tener una curva suave y buena precisión en los picos
omegas_test = np.linspace(0.1, 6.0, 400)
amplitudes_maximas = np.zeros_like(omegas_test)

# 3. Ejecutar el barrido de frecuencias
print("Iniciando barrido de frecuencias...")
for i, w in enumerate(omegas_test):
    # Por cada frecuencia, simulamos la evolución temporal y guardamos la amplitud máxima
    amplitudes_maximas[i] = simular_cuerda_forzada(L, T, mu, N, F_0, w, dt, tiempo_total)

# 4. Identificación automática de las frecuencias resonantes
# 'distance' evita que el algoritmo marque múltiples picos falsos por ruido numérico en la misma campana
indices_picos, _ = find_peaks(amplitudes_maximas, distance=20, prominence=0.001)
omegas_resonantes = omegas_test[indices_picos]
amplitudes_resonantes = amplitudes_maximas[indices_picos]

for i, w_res in enumerate(omegas_resonantes[:3]):
    amp = amplitudes_resonantes[i]
    print(f"Resonancia {i+1}: {w_res:.4f} rad/s con Amplitud Máx = {amp:.4f} m")

In [ ]:
# 5. Construcción de la gráfica
plt.figure(figsize=(10, 6))
plt.plot(omegas_test, amplitudes_maximas, '-', color='b', linewidth=2, label='Respuesta dinámica')
plt.plot(omegas_resonantes, amplitudes_resonantes, 'ro', markersize=8, label='Picos de resonancia')

# Etiquetas y diseño para el reporte
plt.title('Curva de Resonancia: Amplitud máxima vs Frecuencia de excitación', fontsize=14)
plt.xlabel(r'Frecuencia de excitación $\omega$ (rad/s)', fontsize=12)
plt.ylabel('Amplitud máxima alcanzada (m)', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.7)
plt.legend(fontsize=12)
plt.tight_layout()
plt.show()

## 6. Validación con la solución analítica

Se comparan las primeras tres frecuencias de resonancia obtenidas por simulación con la fórmula analítica

$$
\omega_n = \frac{(2n-1)\pi}{2L}\sqrt{\frac{T}{\mu}}
$$

y se calcula el **error relativo porcentual**:

$$
\varepsilon_n = \frac{|\omega_n^{num} - \omega_n^{ana}|}{\omega_n^{ana}} \times 100\%
$$

In [ ]:
# Extraer las primeras 3 frecuencias numéricas de la simulación
omegas_num = omegas_resonantes[:3]

# Calcular las frecuencias analíticas exactas
n_modos = np.array([1, 2, 3])
omegas_ana = ((2 * n_modos - 1) * np.pi / (2 * L)) * np.sqrt(T / mu)

# Comparación y cálculo de error
print("Validación del Modelo (Error Relativo Porcentual):")
print("-" * 50)

# Iterar sobre las frecuencias numéricas extraídas
for i in range(len(omegas_num)):
    w_num = omegas_num[i]
    w_ana = omegas_ana[i]
    error = np.abs(w_num - w_ana) / w_ana * 100

    print(f"Modo {i+1}:")
    print(f"  Numérico (Simulación) = {w_num:.4f} rad/s")
    print(f"  Analítico (Teórico)   = {w_ana:.4f} rad/s")
    print(f"  Error Relativo        = {error:.2f} %")
    print("-" * 55)

## 7. Convergencia espacial

Se repite el cálculo de frecuencias naturales (método de Jacobi) para $N = 10, 20, 40, 80$ y se mide cómo disminuye el error relativo de los tres primeros modos al refinar la malla.

In [ ]:
# --- Análisis de Convergencia Espacial ---
N_valores = [10, 20, 40, 80]
errores_m1, errores_m2, errores_m3 = [], [], []

print("Análisis de convergencia:")
print(f"{'N':<5} | {'Error M1 (%)':<15} | {'Error M2 (%)':<15} | {'Error M3 (%)':<15}")
print("-" * 55)

for N_test in N_valores:
    # 1. Calculamos las numéricas para este N
    _, omegas_num_test, _ = resolver_cuerda(L, T, mu, N_test)

    # 2. Frecuencias analíticas (son las mismas siempre)
    n_modos = np.array([1, 2, 3])
    omegas_ana = ((2 * n_modos - 1) * np.pi / (2 * L)) * np.sqrt(T / mu)

    # 3. Cálculo de errores relativos
    e1 = np.abs(omegas_num_test[0] - omegas_ana[0]) / omegas_ana[0] * 100
    e2 = np.abs(omegas_num_test[1] - omegas_ana[1]) / omegas_ana[1] * 100
    e3 = np.abs(omegas_num_test[2] - omegas_ana[2]) / omegas_ana[2] * 100

    errores_m1.append(e1)
    errores_m2.append(e2)
    errores_m3.append(e3)

    print(f"{N_test:<5} | {e1:<15.4f} | {e2:<15.4f} | {e3:<15.4f}")

In [ ]:
# --- Gráfica de Convergencia ---
plt.figure(figsize=(8, 5))
plt.plot(N_valores, errores_m1, 'o-', lw=2, label='Modo 1')
plt.plot(N_valores, errores_m2, 's-', lw=2, label='Modo 2')
plt.plot(N_valores, errores_m3, '^-', lw=2, label='Modo 3')

plt.title('Convergencia Espacial: Error Numérico vs N', fontsize=14)
plt.xlabel('Número de nodos espaciales (N)', fontsize=12)
plt.ylabel('Error Relativo Porcentual (%)', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.7)
plt.legend(fontsize=12)
plt.tight_layout()
plt.show()

## 8. Evolución temporal y forma de la cuerda en resonancia

### 8.1 Historial completo de la simulación

`simular_historial` es una variante del solver que **guarda el estado de toda la cuerda en cada instante**, produciendo una matriz de tamaño $(\text{pasos}\times N+1)$. Se calcula el historial para las tres primeras resonancias.

In [ ]:
def simular_historial(L, T, mu, N, F_0, omega, dt, tiempo_total):
    dx = L / N
    C = (dt / dx) * np.sqrt(T / mu)
    pasos_temporales = int(tiempo_total / dt)

    # Matriz para almacenar el historial: filas = tiempo, columnas = posición x
    u_historial = np.zeros((pasos_temporales, N + 1))

    u_pasado = np.zeros(N + 1)
    u_presente = np.zeros(N + 1)
    u_futuro = np.zeros(N + 1)

    for n in range(pasos_temporales):
        t = n * dt

        # Nodos interiores
        u_futuro[1:-1] = (2 * u_presente[1:-1] - u_pasado[1:-1] +
                          C**2 * (u_presente[2:] - 2 * u_presente[1:-1] + u_presente[:-2]))

        # Extremo fijo
        u_futuro[0] = 0.0

        # Extremo libre forzado
        u_futuro[N] = (2 * u_presente[N] - u_pasado[N] +
                       C**2 * (2 * u_presente[N-1] - 2 * u_presente[N] +
                               (2 * dx * F_0 / T) * np.cos(omega * t)))

        # Guardar la "fotografía" actual en el historial
        u_historial[n, :] = u_futuro[:]

        # Avanzar en el tiempo
        u_pasado[:] = u_presente[:]
        u_presente[:] = u_futuro[:]

    return u_historial

In [ ]:
# Vectores de referencia
x_nodos = np.linspace(0, L, N + 1)
tiempo = np.linspace(0, tiempo_total, int(tiempo_total / dt))

# Extraer los historiales de los primeros 3 modos resonantes
historiales = []
for w in omegas_num:
    hist = simular_historial(L, T, mu, N, F_0, w, dt, tiempo_total)
    historiales.append(hist)

### 8.2 Evolución temporal del extremo libre

En resonancia la amplitud del extremo libre crece de manera sostenida (sin amortiguamiento) con el tiempo.

In [ ]:
# --- Gráfica de evolución temporal (Actividad 9) ---
num_modos_detectados = len(omegas_num)
fig_time, axs_time = plt.subplots(num_modos_detectados, 1, figsize=(10, 2.5 * num_modos_detectados), sharex=True)
if num_modos_detectados == 1:
    axs_time = [axs_time]

for i in range(num_modos_detectados):
    # Ploteamos solo la última columna (el nodo N, el extremo libre)
    axs_time[i].plot(tiempo, historiales[i][:, -1], color='b')
    axs_time[i].set_ylabel(f'Modo {i+1}\nAmplitud (m)')
    axs_time[i].grid(True, linestyle='--')

axs_time[-1].set_xlabel('Tiempo (s)')
axs_time[0].set_title('Evolución temporal del extremo libre en resonancia')
plt.tight_layout()
plt.show()

### 8.3 Forma espacial de la cuerda

Se captura el perfil $u(x)$ en el instante de máxima elongación del extremo libre (segunda mitad de la simulación). Debe parecerse al **modo normal** correspondiente: 1/4 de longitud de onda para el modo 1, 3/4 para el modo 2, 5/4 para el modo 3.

In [ ]:
# --- Forma estática de la cuerda (Actividad 8) ---
fig_shape, axs_shape = plt.subplots(1, num_modos_detectados, figsize=(5 * num_modos_detectados, 4))
if num_modos_detectados == 1:
    axs_shape = [axs_shape]

for i in range(num_modos_detectados):
    # Buscar en la segunda mitad de la simulación el instante donde el extremo es máximo
    mitad = int(len(tiempo) / 2)
    idx_max = mitad + np.argmax(np.abs(historiales[i][mitad:, -1]))

    # Graficar el perfil espacial de la cuerda en ese instante exacto
    axs_shape[i].plot(x_nodos, historiales[i][idx_max, :], color='r', marker='o')
    axs_shape[i].set_title(f'Modo {i+1} ($\\omega$ = {omegas_num[i]:.2f} rad/s)')
    axs_shape[i].set_xlabel('Posición x (m)')
    axs_shape[i].set_ylabel('Amplitud u(x)')
    axs_shape[i].grid(True, linestyle='--')
plt.tight_layout()
plt.show()

## 9. Animación

Animación del movimiento de la cuerda excitada a la frecuencia de resonancia elegida. Cambia `modo_a_animar` (`0`, `1` o `2`) para ver el modo 1, 2 o 3.


In [ ]:
from IPython.display import HTML

modo_a_animar = 0 # 0 para Modo 1, 1 para Modo 2, 2 para Modo 3
hist_anim = historiales[modo_a_animar]

fig_anim, ax_anim = plt.subplots(figsize=(8, 4))
ax_anim.set_xlim(0, L)
limite_y = np.max(np.abs(hist_anim)) * 1.1
ax_anim.set_ylim(-limite_y, limite_y)
ax_anim.grid(True, linestyle='--')
ax_anim.set_title(f'Animación Modo {modo_a_animar+1} ($\omega$ = {omegas_num[modo_a_animar]:.2f} rad/s)')
ax_anim.set_xlabel('Posición x (m)')
ax_anim.set_ylabel('Amplitud (m)')

linea, = ax_anim.plot([], [], lw=2, color='g', marker='o')

def init():
    linea.set_data([], [])
    return linea,

# Renderizamos 1 de cada 20 cuadros para que el video no pese tanto
salto_frames = 20
def animate(n):
    linea.set_data(x_nodos, hist_anim[n * salto_frames, :])
    return linea,

# Creación de la animación
ani = FuncAnimation(fig_anim, animate, init_func=init,
                    frames=len(tiempo) // salto_frames, interval=20, blit=True)

# Visualizar en Jupyter Notebook
plt.close(fig_anim) # Evita que se imprima una imagen estática residual
HTML(ani.to_jshtml())

## 10. Influencia del tiempo total de simulación

Sin amortiguamiento, en resonancia exacta la amplitud no se estabiliza: crece **linealmente con el tiempo**. La predicción analítica es

$$
A_{max}(t) = \frac{F_0\,t}{\mu\,L\,\omega_n}
$$

### 10.1 Amplitud en resonancia vs. tiempo total

Se simula cada una de las tres resonancias exactas del modelo discreto ($N=10$, obtenidas por Jacobi) para distintos tiempos totales y se compara con la predicción teórica.

In [ ]:
# ===== Verificación 1.7.4: influencia del tiempo total de simulación =====
# Usa: simular_cuerda_forzada, resolver_cuerda (o frecuencias de Jacobi), L, T, mu, N, dt (del notebook) y F_0 = 0.001
import numpy as np
import matplotlib.pyplot as plt

F_0 = 0.001
omegas_jacobi = np.array([1.1096, 3.3014, 5.4120])        # resonancias exactas del modelo con N = 10 (Jacobi)
tiempos = [10, 20, 30, 60, 90, 120]

# --- (a) Amplitud máxima en resonancia exacta vs tiempo total ---
A = np.array([[simular_cuerda_forzada(L, T, mu, N, F_0, w, dt, tt) for tt in tiempos] for w in omegas_jacobi])
# Predicción analítica (sin amortiguamiento): A_max = F_0 * t / (mu * L * omega_n)
A_teo = np.array([[F_0 * tt / (mu * L * w) for tt in tiempos] for w in omegas_jacobi])

print("Amplitud máxima en resonancia (simulada / teórica) vs tiempo total")
print(f"{'t_total (s)':>12} | " + " | ".join(f"modo {i+1}: sim   teo " for i in range(3)))
for k, tt in enumerate(tiempos):
    print(f"{tt:>12} | " + " | ".join(f"{A[i,k]:.4f} {A_teo[i,k]:.4f}" for i in range(3)))

### 10.2 Forma del pico del modo 1

Al aumentar el tiempo total, el pico de resonancia se vuelve **más alto y más estrecho**, ya que la cuerda tiene más tiempo para acumular energía cerca de $\omega_1$. Se mide la posición y el ancho a media altura.

In [ ]:
# --- (b) Posición y ancho del pico del modo 1 vs tiempo total ---
w_barrido = np.linspace(0.9, 1.3, 81)
tiempos_pico = [15, 30, 60, 120]
curvas = {}
print("\nPico del modo 1 (barrido fino) según el tiempo total:")
for tt in tiempos_pico:
    Ac = np.array([simular_cuerda_forzada(L, T, mu, N, F_0, w, dt, tt) for w in w_barrido])
    curvas[tt] = Ac
    mitad = w_barrido[Ac >= Ac.max() / 2]
    print(f"  t_total = {tt:>3} s: ω_pico = {w_barrido[np.argmax(Ac)]:.3f} rad/s, "
          f"A_max = {Ac.max():.4f} m, ancho a media altura ≈ {mitad.max() - mitad.min():.3f} rad/s")

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
for i in range(3):
    ax[0].plot(tiempos, A[i], 'o', label=f'Modo {i+1} (simulación)', color=f'C{i}')
    ax[0].plot(tiempos, A_teo[i], '-', color=f'C{i}', alpha=.6)
ax[0].set_xlabel('Tiempo total (s)'); ax[0].set_ylabel('Amplitud máxima en resonancia (m)')
ax[0].set_title('La amplitud crece como $t$ (líneas: $F_0 t/(\\mu L \\omega_n)$)'); ax[0].legend(); ax[0].grid(True, ls='--', alpha=.6)
for tt, Ac in curvas.items():
    ax[1].plot(w_barrido, Ac, label=f'$t_{{total}}$ = {tt} s')
ax[1].axvline(omegas_jacobi[0], color='k', ls=':', lw=1)
ax[1].set_xlabel('$\\omega$ (rad/s)'); ax[1].set_ylabel('Amplitud máxima (m)')
ax[1].set_title('Pico del modo 1: más alto y más estrecho al aumentar $t_{total}$'); ax[1].legend(); ax[1].grid(True, ls='--', alpha=.6)
plt.tight_layout(); plt.show()

## 11. Influencia de la fuerza externa $F_0$

El sistema es **lineal**: duplicar $F_0$ debe duplicar la amplitud de la respuesta sin cambiar la posición de los picos. Se superponen las curvas de resonancia para $0.5F_0$, $F_0$ y $2F_0$.

In [ ]:
# --- 1. ANÁLISIS DE F_0: Curvas de resonancia superpuestas ---

# Asumimos que L, T, mu, N, dt, tiempo_total y F_0 ya están definidos arriba en tu Colab.
valores_F0 = [F_0 * 0.5, F_0, F_0 * 2.0]
colores = ['blue', 'green', 'red']

omegas_test = np.linspace(0.1, 6.0, 400)
plt.figure(figsize=(10, 6))

for f0_val, color in zip(valores_F0, colores):
    amplitudes_maximas = np.zeros_like(omegas_test)

    for i, w in enumerate(omegas_test):
        # ¡Aquí usamos tu función intacta sin repetir código!
        amplitudes_maximas[i] = simular_cuerda_forzada(L, T, mu, N, f0_val, w, dt, tiempo_total)

    plt.plot(omegas_test, amplitudes_maximas, '-', color=color, linewidth=2, label=f'F_0 = {f0_val:.4f} N')

# Diseño para el reporte
plt.title('Influencia de la Fuerza Externa ($F_0$) en la Curva de Resonancia', fontsize=14)
plt.xlabel(r'Frecuencia de excitación $\omega$ (rad/s)', fontsize=12)
plt.ylabel('Amplitud máxima alcanzada (m)', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.7)
plt.legend(fontsize=12)
plt.tight_layout()
plt.savefig('analisis_F0.png', dpi=300)
plt.show()

## 12. Estabilidad numérica: condición CFL

El esquema explícito es **condicionalmente estable**. El paso temporal crítico es

$$
\Delta t_{crit} = \frac{\Delta x}{\sqrt{T/\mu}}
$$

Se comparan dos simulaciones: una con $\Delta t$ un 1 % **por debajo** del crítico (estable) y otra un 1 % **por encima** (inestable), donde los errores de redondeo crecen exponencialmente.

`simular_historial_extremo` es una versión auxiliar del solver que **omite la verificación** `C >= 1.0`, para poder observar la divergencia.

In [ ]:
# --- 2. ANÁLISIS DE dt: Inestabilidad Numérica ---

def simular_historial_extremo(L, T, mu, N, F_0, omega, dt, tiempo_total):
    """ Función auxiliar exclusiva para extraer el historial de tiempo
        y saltarnos el ValueError de C >= 1.0 """
    dx = L / N
    C = (dt / dx) * np.sqrt(T / mu)
    pasos_temporales = int(tiempo_total / dt)

    u_pasado = np.zeros(N + 1)
    u_presente = np.zeros(N + 1)
    u_futuro = np.zeros(N + 1)

    # Arreglos para guardar la historia
    historial = np.zeros(pasos_temporales)
    t_array = np.zeros(pasos_temporales)

    for n in range(pasos_temporales):
        t = n * dt
        t_array[n] = t
        historial[n] = u_presente[N]

        # Ecuaciones (corregidos los '*' de tu código)
        u_futuro[1:-1] = (2*u_presente[1:-1] - u_pasado[1:-1] +
                          C**2 * (u_presente[2:] - 2*u_presente[1:-1] + u_presente[:-2]))
        u_futuro[0] = 0.0
        u_futuro[N] = (2*u_presente[N] - u_pasado[N] +
                       C**2 * (2*u_presente[N-1] - 2*u_presente[N] + (2*dx * F_0 / T)*np.cos(omega * t)))

        u_pasado[:] = u_presente[:]
        u_presente[:] = u_futuro[:]

    return t_array, historial

In [ ]:
# Calculamos límites
dx_val = L / N
dt_critico = dx_val / np.sqrt(T / mu)

dt_estable = dt_critico * 0.99  # 1% por debajo (Estable)
dt_inestable = dt_critico * 1.01 # 1% por encima (Explotará)

# Usamos un tiempo corto y una frecuencia cualquiera para ver el fallo rápido
omega_test = 2.0
t_corto = 15.0

t_est, ext_est = simular_historial_extremo(L, T, mu, N, F_0, omega_test, dt_estable, t_corto)
t_inest, ext_inest = simular_historial_extremo(L, T, mu, N, F_0, omega_test, dt_inestable, t_corto)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(t_est, ext_est, color='blue')
ax1.set_title(f'Simulación Estable (dt = {dt_estable:.5f} s)')
ax1.set_xlabel('Tiempo (s)')
ax1.set_ylabel('Desplazamiento del Extremo Libre (m)')
ax1.grid(True)

ax2.plot(t_inest, ext_inest, color='red')
ax2.set_title(f'Inestabilidad Numérica (dt = {dt_inestable:.5f} s)')
ax2.set_xlabel('Tiempo (s)')
ax2.grid(True)

plt.tight_layout()
plt.savefig('analisis_dt.png', dpi=300)
plt.show()